In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import shap
import xgboost
from sklearn.linear_model import LinearRegression

In [ ]:
N = 2_000
X = np.zeros((N, 5))

X[:1_000, 0] = 1

X[:500, 1] = 1
X[1_000:1_500, 1] = 1

X[:250, 2] = 1
X[500:750, 2] = 1
X[1_000:1_250, 2] = 1
X[1_500:1_750, 2] = 1

# mean-center the data
X[:, 0:3] -= 0.5

y = 2 * X[:, 0] - 3 * X[:, 1]

In [ ]:
X

In [ ]:
y

In [ ]:
sns.heatmap(X, cmap="bwr")

In [ ]:
sns.lineplot(y)

In [ ]:
np.cov(X.T)

In [ ]:
X.mean(axis=0)

### Simple XGBoost model

In [ ]:
# train a model with single tree
Xd = xgboost.DMatrix(X, label=y)
model = xgboost.train({"eta": 1, "max_depth": 3, "base_score": 0, "lambda": 0}, Xd, 1)

print("Model error =", np.linalg.norm(y - model.predict(Xd)))
print(model.get_dump(with_stats=True)[0])

In [ ]:
pred = model.predict(Xd, output_margin=True)

explainer = shap.TreeExplainer(model)
explanation = explainer(Xd)

shap_values = explanation.values
# make sure the SHAP values add up to marginal predictions
np.abs(shap_values.sum(axis=1) + explanation.base_values - pred).max()

In [ ]:
shap.plots.beeswarm(explanation)

### Linear regression

In [ ]:
lr = LinearRegression()
lr.fit(X, y)
lr_pred = lr.predict(X)
lr.coef_.round(2)

In [ ]:
shap_values

In [ ]:
lr.coef_ * (X - X.mean(0))

In [ ]:
main_effect_shap_values = lr.coef_ * (X - X.mean(0))
np.linalg.norm(shap_values - main_effect_shap_values)

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(Xd)
shap_interaction_values[0]

In [ ]:
np.abs(shap_interaction_values.sum((1, 2)) + explainer.expected_value - pred).max()

In [ ]:
main_effect_shap_values[0]

In [ ]:
total = 0
for i in range(N):
    for j in range(5):
        total += np.abs(
            shap_interaction_values[i, j, j] - main_effect_shap_values[i, j]
        )
total

### Explain linear model with single interaction

In [ ]:
N = 2_000
X = np.zeros((N, 5))
X[:1_000, 0] = 1

X[:500, 1] = 1
X[1_000:1_500, 1] = 1

X[:250, 2] = 1
X[500:750, 2] = 1
X[1_000:1_250, 2] = 1
X[1_500:1_750, 2] = 1

X[:125, 3] = 1
X[250:375, 3] = 1
X[500:625, 3] = 1
X[750:875, 3] = 1
X[1_000:1_125, 3] = 1
X[1_250:1_375, 3] = 1
X[1_500:1_625, 3] = 1
X[1_750:1_875, 3] = 1

# we can't exactly mean center the data or XGBoost has trouble finding the splits
X[:, :4] -= 0.4999

# interaction of features is implemented as the multiplication of the features. Note that any other function of the
#  features would also work, but is harder to interpret (e.g. sin(x1*x2)).
y = 2 * X[:, 0] - 3 * X[:, 1] + 2 * X[:, 1] * X[:, 2]

In [ ]:
sns.heatmap(X, cmap="bwr")

In [ ]:
sns.lineplot(y)

In [ ]:
X.mean(axis=0)

In [ ]:
# train a model with single tree
Xd = xgboost.DMatrix(X, label=y)
model = xgboost.train({"eta": 1, "max_depth": 4, "base_score": 0, "lambda": 0}, Xd, 1)
print("Model error =", np.linalg.norm(y - model.predict(Xd)))
print(model.get_dump(with_stats=True)[0])

In [ ]:
# plot the first tree in the trained booster
fig, ax = plt.subplots(figsize=(18, 6))
xgboost.plot_tree(model, num_trees=0, ax=ax)
ax.set_title("XGBoost tree 0")
plt.show()

In [ ]:
pred = model.predict(Xd, output_margin=True)

explainer = shap.TreeExplainer(model)
explanation = explainer(Xd)

shap_values = explanation.values
# make sure the SHAP values add up to marginal predictions
np.abs(shap_values.sum(axis=1) + explanation.base_values - pred).max()

In [ ]:
shap.plots.beeswarm(explanation)

In [ ]:
lr = LinearRegression()
lr.fit(X, y)
lr_pred = lr.predict(X)
lr.coef_.round(2)

In [ ]:
main_effect_shap_values = lr.coef_ * (X - X.mean(0))
np.linalg.norm(shap_values - main_effect_shap_values)

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(Xd)
shap_interaction_values[0].round(2)

In [ ]:
np.abs(shap_interaction_values.sum((1, 2)) + explainer.expected_value - pred).max()

In [ ]:
total = 0
for i in range(N):
    for j in range(5):
        total += np.abs(
            shap_interaction_values[i, j, j] - main_effect_shap_values[i, j]
        )
total

In [ ]:
shap.dependence_plot(0, shap_values, X)

In [ ]:
shap.dependence_plot(2, shap_values, X)